# Manual notebook secret verification

Run this notebook in each hosted runtime after installing the Omniframes version under test.
It reads secrets through the session builder and prints status only. This notebook is an
**unexecuted manual check**, not evidence of live platform validation.

Prepare both a URL secret and an API-key secret. Use only secret names in the configuration
cell; never paste credential values here. Follow the
[platform setup instructions](https://github.com/exploreomni/omniframes/blob/main/docs/quickstart.md#notebook-secrets)
for Colab notebook access, Databricks scopes, or Snowflake secret attachments and external access.

- **Colab:** use `PLATFORM = "colab"`, default secret names, and `USE_AUTO_DETECTION = True`.
- **Databricks:** use `PLATFORM = "databricks"`, set `SECRET_SCOPE`, and keep automatic detection
  enabled to exercise the notebook's existing `dbutils` object.
- **Snowflake Workspaces:** use `PLATFORM = "snowflake"` and set both secret names to their
  normalized `database/schema/name` paths.
- **Legacy Snowflake:** use `PLATFORM = "snowflake-legacy"` and set both names to notebook aliases.

Snowflake always uses explicit selection. After a successful Colab or Databricks run, set
`USE_AUTO_DETECTION = False` and rerun to check explicit provider selection too.


In [ ]:
import os

from omniframes import OmniSession

PLATFORM = "colab"
SECRET_SCOPE = None  # Databricks only: replace with your scope name.
API_KEY_SECRET = "OMNI_API_KEY"
BASE_URL_SECRET = "OMNI_BASE_URL"
USE_AUTO_DETECTION = True
VERIFY_OMNI = False  # Opt in separately to sending the credentials to Omni.

## Read through the builder

This guard prevents environment credentials from masking the secret lookup being tested.
If it fails, remove those variables from this notebook's environment before rerunning; it
never changes or displays environment values. No explicit credential or fake transport is
provided, so successful construction requires both secrets to resolve.


In [ ]:
if any(name in os.environ for name in ("OMNI_BASE_URL", "OMNI_API_KEY")):
    raise RuntimeError(
        "Unset OMNI_BASE_URL and OMNI_API_KEY in this notebook environment "
        "before testing secret resolution."
    )

if PLATFORM not in {"colab", "databricks", "snowflake", "snowflake-legacy"}:
    raise ValueError("Select one of the four documented notebook platforms.")
if PLATFORM == "databricks" and not SECRET_SCOPE:
    raise ValueError("Set SECRET_SCOPE to the Databricks scope containing both secrets.")

provider = "auto" if USE_AUTO_DETECTION and PLATFORM in {"colab", "databricks"} else PLATFORM
builder = OmniSession.builder.secrets(
    provider,
    scope=SECRET_SCOPE,
    api_key_name=API_KEY_SECRET,
    base_url_name=BASE_URL_SECRET,
)
session = builder.get_or_create()
print("PASS: session built from notebook secrets; no Omni API call made.")

## Optional Omni authentication check

Leave `VERIFY_OMNI = False` to stop at secret resolution. Enable it only after confirming the
URL secret points at your Omni organization. Snowflake needs an EAI network rule permitting
that hostname. The check calls `session.verify()` and prints no identity or credential data.


In [ ]:
try:
    if VERIFY_OMNI:
        session.verify()
        print("PASS: Omni accepted the notebook credentials.")
    else:
        print("SKIPPED: live Omni authentication was not requested.")
finally:
    session.close()

## Record the result

Record the platform/runtime version, Omniframes version, date, and whether automatic or
explicit selection passed. Keep credentials and identity responses out of the record.

To inspect failures, rerun the lookup cell with an intentionally absent secret name. Where
possible, separately revoke this notebook's access to a dedicated test secret and rerun.
Restore the original configuration afterward. Expect an actionable, sanitized error; some
providers cannot distinguish a missing secret from one the notebook cannot access.

A passing construction check verifies secret retrieval only. It does not prove Omni
permissions or network connectivity unless you separately enabled `VERIFY_OMNI`.
